BAUDE-45


In [0]:
%sql

-- =============================================================================
-- DROP ALL pipeline logging objects (every naming iteration) — EXCEPT expected_tasks
-- Views first (they depend on tables), then tables.  Replace {env} with 'p'.
-- =============================================================================


-- ---- VIEWS ----
DROP VIEW  IF EXISTS `p-cdh`.pipeline.v_task_status            -- 1st iteration
;DROP VIEW  IF EXISTS `p-cdh`.pipeline.v_job_task_log           -- 2nd iteration
;DROP VIEW  IF EXISTS `p-cdh`.pipeline.v_latest_execution       -- old area view
;DROP VIEW  IF EXISTS `p-cdh`.pipeline.v_task_latest_status     -- final (recreate after)
;DROP VIEW  IF EXISTS `p-cdh`.pipeline.v_area_latest_status     -- final (recreate after)

-- ---- TABLES ----
;DROP TABLE IF EXISTS `p-cdh`.pipeline.job_task_log            -- 1st iteration
;DROP TABLE IF EXISTS `p-cdh`.pipeline.job_task_events          -- 2nd iteration
;DROP TABLE IF EXISTS `p-cdh`.pipeline.job_execution_log        -- old area table
;DROP TABLE IF EXISTS `p-cdh`.pipeline.task_runs                -- final (recreate after)
;DROP TABLE IF EXISTS `p-cdh`.pipeline.area_runs                -- final (recreate after)




;CREATE TABLE `p-cdh`.pipeline.task_runs (
    pipeline_area       STRING          NOT NULL    COMMENT 'Area: staging, landing, landed, curated, derivative',
    business_date       DATE            NOT NULL    COMMENT 'Business date of the data being processed',
    schema_name         STRING                      COMMENT 'Source schema (start event only)',
    group_name          STRING                      COMMENT 'Group within schema (start event only)',
    task_name           STRING          NOT NULL    COMMENT 'What was processed (e.g., profits_loan, DExposure)',
    event_type          STRING          NOT NULL    COMMENT 'start | completed | failed',
    databricks_run_id   STRING                      COMMENT 'Databricks job run ID',
    job_name            STRING                      COMMENT 'Databricks job name',
    error_message       STRING                      COMMENT 'Error details (failed events only)',
    triggered_by        STRING                      COMMENT 'Notebook that called the function (auto-detected)',
    event_timestamp     TIMESTAMP       NOT NULL    COMMENT 'When this event happened (the only timestamp)'
)
USING DELTA
PARTITIONED BY (business_date)

;CREATE OR REPLACE VIEW `p-cdh`.pipeline.v_task_latest_status AS
WITH latest_event AS (
    -- the most recent event per task = current state
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY pipeline_area, business_date, task_name
            ORDER BY event_timestamp DESC
        ) AS rn
    FROM `p-cdh`.pipeline.task_runs
),
cur AS (
    SELECT * FROM latest_event WHERE rn = 1
),
end_cycle AS (
    -- the latest terminal event (completed/failed) per task = end of last cycle
    SELECT pipeline_area, business_date, task_name, event_timestamp AS end_ts,
        ROW_NUMBER() OVER (
            PARTITION BY pipeline_area, business_date, task_name
            ORDER BY event_timestamp DESC
        ) AS ern
    FROM `p-cdh`.pipeline.task_runs
    WHERE event_type IN ('completed', 'failed')
),
end_latest AS (
    SELECT pipeline_area, business_date, task_name, end_ts
    FROM end_cycle WHERE ern = 1
),
start_cycle AS (
    -- the latest 'start' at or before that end = start of last cycle
    SELECT e.pipeline_area, e.business_date, e.task_name, e.end_ts,
        (SELECT MAX(s.event_timestamp)
         FROM `p-cdh`.pipeline.task_runs s
         WHERE s.pipeline_area = e.pipeline_area
           AND s.business_date = e.business_date
           AND s.task_name     = e.task_name
           AND s.event_type    = 'start'
           AND s.event_timestamp <= e.end_ts) AS start_ts
    FROM end_latest e
)
SELECT
    cur.pipeline_area,
    cur.business_date,
    cur.schema_name,
    cur.group_name,
    cur.task_name,
    CASE cur.event_type
        WHEN 'start'     THEN 'running'
        WHEN 'completed' THEN 'completed'
        WHEN 'failed'    THEN 'failed'
        ELSE cur.event_type
    END                                           AS status,
    sc.start_ts                                   AS start_timestamp,
    -- only show end/duration when the task is NOT currently running
    CASE WHEN cur.event_type = 'start' THEN NULL ELSE sc.end_ts END               AS end_timestamp,
    CASE WHEN cur.event_type = 'start' THEN NULL
         ELSE CAST(unix_timestamp(sc.end_ts) - unix_timestamp(sc.start_ts) AS INT)
    END                                            AS duration_seconds,
    cur.databricks_run_id,
    cur.job_name,
    cur.error_message,
    cur.triggered_by
FROM cur
LEFT JOIN start_cycle sc
    ON  sc.pipeline_area = cur.pipeline_area
    AND sc.business_date = cur.business_date
    AND sc.task_name     = cur.task_name
    
    
;CREATE TABLE `p-cdh`.pipeline.area_runs (
    pipeline_area       STRING          NOT NULL    COMMENT 'Area: landing, curated, derivative, ...',
    business_date       DATE            NOT NULL    COMMENT 'Business date of the data being processed',
    status              STRING          NOT NULL    COMMENT 'completed (success-only for now)',
    start_timestamp     TIMESTAMP                   COMMENT 'Area start = MIN of task starts (last cycle)',
    end_timestamp       TIMESTAMP                   COMMENT 'Area end = MAX of task ends (last cycle)',
    duration_seconds    INT                         COMMENT 'Total elapsed seconds',
    triggered_by        STRING                      COMMENT 'auto_completion_check | ...',
    databricks_run_id   STRING                      COMMENT 'Databricks job run ID',
    error_message       STRING                      COMMENT 'Error details (null if success)',
    logged_at           TIMESTAMP       NOT NULL    COMMENT 'When this record was written'
)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'Area completion log — one success record per area per business_date.'
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
)


;CREATE OR REPLACE VIEW `p-cdh`.pipeline.v_area_latest_status AS
SELECT
    pipeline_area,
    business_date,
    status,
    start_timestamp,
    end_timestamp,
    duration_seconds,
    triggered_by,
    databricks_run_id,
    error_message,
    logged_at
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY pipeline_area
            ORDER BY business_date DESC, logged_at DESC
        ) AS rn
    FROM `p-cdh`.pipeline.area_runs
    WHERE status = 'completed'
)
WHERE rn = 1


BAUDE-122


In [0]:
%sql
CREATE OR REPLACE FUNCTION `p-cdh`.derivative.get_exposures_by_business_date(
  business_date DATE,
  include_migrated BOOLEAN DEFAULT false
)
RETURNS TABLE
RETURN
WITH
latest AS (
    SELECT ExposureKey, MAX(BusinessDate) AS MaxBD
    FROM (
        SELECT ExposureKey, BusinessDate
        FROM `p-cdh`.derivativeactive.dexposure
        WHERE BusinessDate <= business_date
        UNION ALL
        SELECT ExposureKey, BusinessDate
        FROM `p-cdh`.derivativearchive.dexposure
        WHERE BusinessDate <= business_date
          AND BusinessDate < DATE'2024-01-30'
    ) combined
    GROUP BY ExposureKey
)

SELECT B.*
FROM latest A
INNER JOIN `p-cdh`.derivativeactive.dexposure B
  ON A.ExposureKey = B.ExposureKey
  AND A.MaxBD = B.BusinessDate
WHERE A.MaxBD >= DATE'2024-01-30'
  AND (
    (B.IsActiveFlagInt = 1 AND B.BusinessDate = business_date)
    OR (include_migrated OR COALESCE(B.JustificationCode, 0) != 43023)
  )

UNION ALL

SELECT B.*
FROM latest A
INNER JOIN `p-cdh`.derivativearchive.dexposure B
  ON A.ExposureKey = B.ExposureKey
  AND A.MaxBD = B.BusinessDate
WHERE A.MaxBD < DATE'2024-01-30'
  AND (
    (B.IsActiveFlagInt = 1 AND B.BusinessDate = business_date)
    OR (include_migrated OR COALESCE(B.JustificationCode, 0) != 43023)
  );